# Actions and Transformation in PySpark

## Initialization

In [2]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
sc = SparkContext.getOrCreate()
spark = SparkSession.builder.appName('Actions & Transformations').getOrCreate()

## Uploading dataset and reading

In [6]:
df = spark.read.csv("/content/loan.csv", header=True, inferSchema=True)
df.show(5)


+-----------+---+------+------------+--------------+-----------+------+-----------+-------------+-------------+-----------+-------+------------+----------------+------------------+
|Customer_ID|Age|Gender|  Occupation|Marital Status|Family Size|Income|Expenditure|Use Frequency|Loan Category|Loan Amount|Overdue| Debt Record| Returned Cheque| Dishonour of Bill|
+-----------+---+------+------------+--------------+-----------+------+-----------+-------------+-------------+-----------+-------+------------+----------------+------------------+
|    IB14001| 30|  MALE|BANK MANAGER|        SINGLE|          4| 50000|      22199|            6|      HOUSING| 10,00,000 |      5|      42,898|               6|                 9|
|    IB14008| 44|  MALE|   PROFESSOR|       MARRIED|          6| 51000|      19999|            4|     SHOPPING|     50,000|      3|      33,999|               1|                 5|
|    IB14012| 30|FEMALE|     DENTIST|        SINGLE|          3| 58450|      27675|            

## Transformations

### 1.Filter

In [7]:
high_income_df = df.filter(df["Income"] > 50000)
high_income_df.show(5)


+-----------+---+------+-------------------+--------------+-----------+------+-----------+-------------+----------------+-----------+-------+------------+----------------+------------------+
|Customer_ID|Age|Gender|         Occupation|Marital Status|Family Size|Income|Expenditure|Use Frequency|   Loan Category|Loan Amount|Overdue| Debt Record| Returned Cheque| Dishonour of Bill|
+-----------+---+------+-------------------+--------------+-----------+------+-----------+-------------+----------------+-----------+-------+------------+----------------+------------------+
|    IB14008| 44|  MALE|          PROFESSOR|       MARRIED|          6| 51000|      19999|            4|        SHOPPING|     50,000|      3|      33,999|               1|                 5|
|    IB14012| 30|FEMALE|            DENTIST|        SINGLE|          3| 58450|      27675|            5|      TRAVELLING|     75,000|      6|      20,876|               3|                 1|
|    IB14031| 37|FEMALE|  SOFTWARE ENGINEER| 

In [9]:
print(df.columns)


['Customer_ID', 'Age', 'Gender', 'Occupation', 'Marital Status', 'Family Size', 'Income', 'Expenditure', 'Use Frequency', 'Loan Category', 'Loan Amount', 'Overdue', ' Debt Record', ' Returned Cheque', ' Dishonour of Bill']


###  2. Join

In [15]:
from pyspark.sql.functions import col

# First clean up column names
for c in df.columns:
    df = df.withColumnRenamed(c, c.strip().replace(" ", "_"))

In [16]:
# Create aliases for self join
df_a = df.alias("a")
df_b = df.alias("b")

In [18]:
# # Perform join on Occupation
join_df = df_a.join(
      df_b,
      (col("a.Occupation") == col("b.Occupation")) &
      (col("a.Customer_ID") != col("b.Customer_ID"))
).select(
       col("a.Customer_ID").alias("Customer_A"),
       col("b.Customer_ID").alias("Customer_B"),
       col("a.Occupation"),
       col("a.Income").alias("Income_A"),
       col("b.Income").alias("Income_B")
)

join_df.show(10)



+----------+----------+------------+--------+--------+
|Customer_A|Customer_B|  Occupation|Income_A|Income_B|
+----------+----------+------------+--------+--------+
|   IB14001|   IB15104|BANK MANAGER|   50000|    NULL|
|   IB14001|   IB15077|BANK MANAGER|   50000|   68125|
|   IB14001|   IB14960|BANK MANAGER|   50000|   58697|
|   IB14001|   IB14955|BANK MANAGER|   50000|   63070|
|   IB14001|   IB14926|BANK MANAGER|   50000|   60740|
|   IB14001|   IB14886|BANK MANAGER|   50000|   85632|
|   IB14001|   IB14864|BANK MANAGER|   50000|   32571|
|   IB14001|   IB14827|BANK MANAGER|   50000|   35735|
|   IB14001|   IB14685|BANK MANAGER|   50000|   29565|
|   IB14001|   IB14655|BANK MANAGER|   50000|   68918|
+----------+----------+------------+--------+--------+
only showing top 10 rows



### 3. Simple Aggregation

In [19]:
from pyspark.sql.functions import avg

avg_exp = df.agg(avg("Expenditure"))
avg_exp.show()


+------------------+
|  avg(Expenditure)|
+------------------+
|27533.180873180874|
+------------------+



###  4. GroupBy

In [22]:
from pyspark.sql.functions import regexp_replace, col

# Remove commas (e.g., "1,00,000" => "100000") and cast to float
df = df.withColumn("Loan_Amount_Clean", regexp_replace("Loan_Amount", ",", "").cast("float"))


In [23]:
grouped_df = df.groupBy("Loan_Category").sum("Loan_Amount_Clean")
grouped_df.show()


+------------------+----------------------+
|     Loan_Category|sum(Loan_Amount_Clean)|
+------------------+----------------------+
|           HOUSING|           6.0346129E7|
|        TRAVELLING|           3.6608973E7|
|       BOOK STORES|             3681651.0|
|       AGRICULTURE|           1.1590221E7|
|         GOLD LOAN|           7.0991425E7|
|  EDUCATIONAL LOAN|           1.8394223E7|
|        AUTOMOBILE|           5.6542964E7|
|          BUSINESS|           2.3368358E7|
|COMPUTER SOFTWARES|           3.4810861E7|
|           DINNING|             9167850.0|
|          SHOPPING|           1.5645414E7|
|       RESTAURANTS|           2.5006754E7|
|       ELECTRONICS|             8970419.0|
|          BUILDING|             3792037.0|
|        RESTAURANT|           1.1729243E7|
|   HOME APPLIANCES|             7879927.0|
+------------------+----------------------+



###  5. Window Functions

In [24]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

window_spec = Window.orderBy(df["Income"].desc())
ranked_df = df.withColumn("Income_Rank", rank().over(window_spec))
ranked_df.select("Customer_ID", "Income", "Income_Rank").show()


+-----------+------+-----------+
|Customer_ID|Income|Income_Rank|
+-----------+------+-----------+
|    IBI4157|930000|          1|
|    IB14107|800000|          2|
|    IB14163|800000|          2|
|    IB14256|800000|          2|
|    IB14128|750000|          5|
|    IB14249|700000|          6|
|    IB14968|420792|          7|
|    IB14557| 98640|          8|
|    IB14681| 98586|          9|
|    IB14699| 98199|         10|
|    IB14641| 96681|         11|
|    IB14889| 96501|         12|
|    IB14991| 95853|         13|
|    IB14652| 95789|         14|
|    IB15024| 95425|         15|
|    IB14390| 95247|         16|
|    IB14620| 93474|         17|
|    IB14716| 92425|         18|
|    IB14585| 91853|         19|
|    IB14867| 91088|         20|
+-----------+------+-----------+
only showing top 20 rows



## Actions

### 1. collect()

In [33]:
all_rows = df.collect()[:10]  # Limit to 10 to avoid memory overload
for row in all_rows:
    print(", ".join([str(x) for x in row]))


IB14001, 30, MALE, BANK MANAGER, SINGLE, 4, 50000, 22199, 6, HOUSING,  10,00,000 , 5, 42,898, 6, 9, 1000000.0
IB14008, 44, MALE, PROFESSOR, MARRIED, 6, 51000, 19999, 4, SHOPPING, 50,000, 3, 33,999, 1, 5, 50000.0
IB14012, 30, FEMALE, DENTIST, SINGLE, 3, 58450, 27675, 5, TRAVELLING, 75,000, 6, 20,876, 3, 1, 75000.0
IB14018, 29, MALE, TEACHER, MARRIED, 5, 45767, 12787, 3, GOLD LOAN,  6,00,000 , 7, 11,000, 0, 4, 600000.0
IB14022, 34, MALE, POLICE, SINGLE, 4, 43521, 11999, 3, AUTOMOBILE,  2,00,000 , 2, 43,898, 1, 2, 200000.0
IB14024, 55, FEMALE, NURSE, MARRIED, 6, 34999, 19888, 4, AUTOMOBILE, 47,787, 1, 50,000, 0, 3, 47787.0
IB14025, 39, FEMALE, TEACHER, MARRIED, 6, 46619, 18675, 4, HOUSING,  12,09,867 , 8, 29,999, 6, 8, 1209867.0
IB14027, 51, MALE, SYSTEM MANAGER, MARRIED, 3, 49999, 19111, 5, RESTAURANTS, 60,676, 8, 13,000, 2, 5, 60676.0
IB14029, 24, FEMALE, TEACHER, SINGLE, 3, 45008, 17454, 4, AUTOMOBILE,  3,99,435 , 9, 51,987, 4, 7, 399435.0
IB14031, 37, FEMALE, SOFTWARE ENGINEER, MARRIE

### 2. count()

In [34]:
total_count = df.count()
print("Total rows:", total_count)


Total rows: 500


### 3. take()

In [35]:
rows = df.take(10)
for row in rows:
    print(", ".join([str(x) for x in row]))


IB14001, 30, MALE, BANK MANAGER, SINGLE, 4, 50000, 22199, 6, HOUSING,  10,00,000 , 5, 42,898, 6, 9, 1000000.0
IB14008, 44, MALE, PROFESSOR, MARRIED, 6, 51000, 19999, 4, SHOPPING, 50,000, 3, 33,999, 1, 5, 50000.0
IB14012, 30, FEMALE, DENTIST, SINGLE, 3, 58450, 27675, 5, TRAVELLING, 75,000, 6, 20,876, 3, 1, 75000.0
IB14018, 29, MALE, TEACHER, MARRIED, 5, 45767, 12787, 3, GOLD LOAN,  6,00,000 , 7, 11,000, 0, 4, 600000.0
IB14022, 34, MALE, POLICE, SINGLE, 4, 43521, 11999, 3, AUTOMOBILE,  2,00,000 , 2, 43,898, 1, 2, 200000.0
IB14024, 55, FEMALE, NURSE, MARRIED, 6, 34999, 19888, 4, AUTOMOBILE, 47,787, 1, 50,000, 0, 3, 47787.0
IB14025, 39, FEMALE, TEACHER, MARRIED, 6, 46619, 18675, 4, HOUSING,  12,09,867 , 8, 29,999, 6, 8, 1209867.0
IB14027, 51, MALE, SYSTEM MANAGER, MARRIED, 3, 49999, 19111, 5, RESTAURANTS, 60,676, 8, 13,000, 2, 5, 60676.0
IB14029, 24, FEMALE, TEACHER, SINGLE, 3, 45008, 17454, 4, AUTOMOBILE,  3,99,435 , 9, 51,987, 4, 7, 399435.0
IB14031, 37, FEMALE, SOFTWARE ENGINEER, MARRIE

### 4. first()

In [36]:
first_row = df.first()
print(", ".join([str(x) for x in first_row]))


IB14001, 30, MALE, BANK MANAGER, SINGLE, 4, 50000, 22199, 6, HOUSING,  10,00,000 , 5, 42,898, 6, 9, 1000000.0


### 5. SaveasTextFile()

In [41]:
# Save as text (each row becomes one line)
df.rdd.map(lambda row: ", ".join([str(x) for x in row])).saveAsTextFile("/content/loan_output")


In [43]:
#  To view the saved output
!cat /content/loan_output/part-00000


IB14001, 30, MALE, BANK MANAGER, SINGLE, 4, 50000, 22199, 6, HOUSING, None, 5, 42,898, 6, 9, 1000000.0
IB14008, 44, MALE, PROFESSOR, MARRIED, 6, 51000, 19999, 4, SHOPPING, None, 3, 33,999, 1, 5, 50000.0
IB14012, 30, FEMALE, DENTIST, SINGLE, 3, 58450, 27675, 5, TRAVELLING, None, 6, 20,876, 3, 1, 75000.0
IB14018, 29, MALE, TEACHER, MARRIED, 5, 45767, 12787, 3, GOLD LOAN, None, 7, 11,000, 0, 4, 600000.0
IB14022, 34, MALE, POLICE, SINGLE, 4, 43521, 11999, 3, AUTOMOBILE, None, 2, 43,898, 1, 2, 200000.0
IB14024, 55, FEMALE, NURSE, MARRIED, 6, 34999, 19888, 4, AUTOMOBILE, None, 1, 50,000, 0, 3, 47787.0
IB14025, 39, FEMALE, TEACHER, MARRIED, 6, 46619, 18675, 4, HOUSING, None, 8, 29,999, 6, 8, 1209867.0
IB14027, 51, MALE, SYSTEM MANAGER, MARRIED, 3, 49999, 19111, 5, RESTAURANTS, None, 8, 13,000, 2, 5, 60676.0
IB14029, 24, FEMALE, TEACHER, SINGLE, 3, 45008, 17454, 4, AUTOMOBILE, None, 9, 51,987, 4, 7, 399435.0
IB14031, 37, FEMALE, SOFTWARE ENGINEER, MARRIED, 5, 55999, 23999, 5, AUTOMOBILE, None,